In [ ]:
# Ensure src/ is on sys.path so top-level imports like `progress` resolve when importing modules
import sys
if '/kaggle/working/src' not in sys.path:
    sys.path.insert(0, '/kaggle/working/src')
print('Inserted /kaggle/working/src into sys.path for Kaggle runtime')

# Kaggle GPU Run: Voice Cloning Tool

This notebook prepares a Kaggle GPU environment, installs minimal dependencies, patches the repo for Kaggle paths, preloads the Coqui TTS model (XTTS-v2), precomputes conditioning latents for a voice sample, runs generation using an optimized pipeline, and measures performance (RTF). Follow the cells in order. This notebook is designed to run non-interactively in a Kaggle GPU session.

## 1) Prep Kaggle environment and verify GPU
Run quick checks to confirm GPU availability and set working directories.

In [ ]:
# Cell: Verify GPU and set working dir
import os
import sys
import time

print('cwd before:', os.getcwd())
# Kaggle working dir
os.chdir('/kaggle/working')
print('cwd now:', os.getcwd())

# GPU checks
try:
    gpu_out = os.popen('nvidia-smi -L').read()
    print(gpu_out.strip() or 'No GPUs detected by nvidia-smi')
except Exception as e:
    print('nvidia-smi not available:', e)

import torch
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count())
print('cuda version:', torch.version.cuda)

# Create cache and output directories
os.environ.setdefault('VC_CACHE', '/kaggle/working/cache')
os.makedirs('/kaggle/working/cache', exist_ok=True)
os.makedirs('/kaggle/working/output', exist_ok=True)
print('Created /kaggle/working/cache and /kaggle/working/output')

## 2) Copy or clone project files into /kaggle/working
If your repo is available via GitHub, you can clone it. Otherwise upload the `voice-clone-tool` folder into the Kaggle Notebook's Files (via 'Add data' -> Upload or use a linked dataset). Example commands are shown below.

In [ ]:
# Cell: Clone repo (optional)
# Uncomment and set your GitHub repo if you want to clone directly
# !git clone https://github.com/<yourusername>/voice-cloning-tool.git /kaggle/working/voice-clone-tool

# If you uploaded a dataset with the code, copy it into working dir. Example:
# !mkdir -p /kaggle/working/voice-clone-tool
# !cp -r /kaggle/input/<your-uploaded-dataset>/voice-clone-tool/* /kaggle/working/voice-clone-tool/

# For this notebook, assume the repository files are already present under the working dir.

import os
print('Files in /kaggle/working:', os.listdir('/kaggle/working')[:20])

# Verify presence of key files
expected = ['src', 'data', 'requirements.txt']
for e in expected:
    print(e, 'exists:', os.path.exists(os.path.join('/kaggle/working', e)))

## 3) Install system packages and Python dependencies
Install ffmpeg and the minimal Python packages required to run Coqui TTS and this project. Installing the full requirements can be slow; try the minimal set first.

In [ ]:
# Cell: Install ffmpeg and minimal Python packages
# On Kaggle, apt-get is available

print('Installing system packages and python dependencies. This may take several minutes...')

# Install system-level packages
try:
    !apt-get update -y && apt-get install -y ffmpeg libsndfile1 sox
except Exception as e:
    print('apt-get may not be available in this environment:', e)

# Upgrade pip and basic build tools
!pip install -q --upgrade pip setuptools wheel

# Detect CUDA and install appropriate PyTorch wheel if needed
import torch
cuda_available = torch.cuda.is_available()
print('Initial torch.cuda.is_available():', cuda_available)

if not cuda_available:
    # Try installing a CUDA-enabled torch wheel compatible with Kaggle's CUDA
    # Kaggle often supports CUDA 11.x or 12.x; try a broadly compatible wheel (may change over time)
    try:
        print('Installing torch with CUDA support (this may take some minutes)...')
        !pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118
    except Exception as e:
        print('Automatic torch install failed, you may need to pick correct CUDA wheel manually:', e)

# Install Coqui TTS minimal packages
!pip install -q coqui-tts==0.27.1 soundfile librosa numpy numba tqdm

print('Dependency installation step finished. Verify versions:')
import importlib
for pkg in ['torch','TTS','librosa','soundfile','numpy']:
    try:
        m = importlib.import_module(pkg)
        print(pkg, 'version:', getattr(m, '__version__', 'n/a'))
    except Exception as e:
        print(pkg, 'import failed:', e)

## 4) Patch repository for Kaggle GPU (idempotent edits)
Make small, safe edits to `src/voice_clone.py` so it uses `/kaggle/working/cache` and guards FP16 conversions. This cell performs in-place edits and backs up the original file.

In [ ]:
# Cell: Patch voice_clone.py for Kaggle paths and safe FP16
from pathlib import Path
import shutil
import re

vc_path = Path('src/voice_clone.py')
backup_path = Path('src/voice_clone.py.bak')
if vc_path.exists() and not backup_path.exists():
    print('Backing up original voice_clone.py to src/voice_clone.py.bak')
    shutil.copy(vc_path, backup_path)

text = vc_path.read_text(encoding='utf-8')

# Replace cache dir assignment to use VC_CACHE env var
text_new = re.sub(r"self\.cache_dir\s*=\s*Path\(['\"]cache['\"]\)",
                  "self.cache_dir = Path(os.getenv('VC_CACHE','/kaggle/working/cache'))",
                  text)

# Ensure os import exists
if 'import os' not in text_new.splitlines()[0:20]:
    text_new = 'import os\n' + text_new

# Guard FP16 conversion: replace .half() calls with guarded version
text_new = text_new.replace(".half()", ".half() if (self.device == 'cuda' and torch.cuda.is_available()) else None")

vc_path.write_text(text_new, encoding='utf-8')
print('Patched src/voice_clone.py: cache dir set to VC_CACHE and FP16 conversions guarded')

## 5) Preload and optimize TTS model (download & warm-up)
Instantiate the cloner, ensure the model loads into GPU, and perform a warm-up call. This speeds up the first generation and caches model weights in `/kaggle/working/cache`.

In [ ]:
# Cell: Load the VoiceClone class and ensure model is loaded/warmed up
import time
from pathlib import Path

# Add repo root to sys.path
import sys
repo_root = Path('/kaggle/working')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.voice_clone import UltraOptimizedVoiceClone

vc = UltraOptimizedVoiceClone()
start = time.time()
vc._ensure_model_loaded_ultra_fast()
# Warmup short phrase
try:
    _ = vc.tts.tts('This is a warm up phrase.', speaker_wav=None, language='en')
except Exception as e:
    print('Warmup via high-level API failed:', e)

print('Model load + warmup time:', time.time() - start)
print('Cache dir used:', vc.cache_dir)

## 6) Configure PyTorch and environment for inference speed
Set flags for inference: disable gradients, enable cuDNN benchmark, and set deterministic to False. Try using autocast for FP16 where possible.

In [ ]:
# Cell: Configure PyTorch for inference
import torch

torch.set_grad_enabled(False)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

print('Configured torch for inference. Amp available:', hasattr(torch.cuda, 'amp'))

# If using PyTorch 2.x, optionally use torch.compile (guarded)
try:
    if hasattr(torch, 'compile'):
        print('torch.compile available; compilation will be attempted during model load if supported')
except Exception as e:
    print('torch.compile check failed:', e)

## 7) Precompute conditioning latents and persistent caching
Upload a clean voice sample to `/kaggle/working/voice_sample.wav` (or use the repo sample) and run precomputation so generation is fast and consistent.

In [ ]:
# Cell: Precompute conditioning latents
from pathlib import Path

# Use repo sample if present
sample_candidates = [
    Path('/kaggle/working/voice_sample.wav'),
    Path('/kaggle/working/data/voice_samples/my_voice.mp3'),
    Path('/kaggle/working/data/voice_samples/my_voice.wav')
]
voice_sample = None
for c in sample_candidates:
    if c.exists():
        voice_sample = c
        break

if voice_sample is None:
    print('No voice sample found; please upload one to /kaggle/working/voice_sample.wav or add to data/voice_samples/')
else:
    print('Using voice sample:', voice_sample)
    start = time.time()
    vc.precompute_conditioning_latents(str(voice_sample))
    print('Precompute time:', time.time()-start)
    print('Current gpt_cond set:', vc.current_gpt_cond_latent is not None)
    print('Cache files:', list(Path('/kaggle/working/cache').glob('*.pkl')))

## 8) Ultra-intelligent chunking and generation pipeline
Use the cloner's chunking and generation helpers to synthesize the script with minimal overhead and measure per-chunk timings.

In [ ]:
# Cell: Run generation pipeline on repo script (if present)
from pathlib import Path
import soundfile as sf
import numpy as np

script_path = Path('/kaggle/working/data/script.txt')
if not script_path.exists():
    # fallback to short example text
    text = "Hello. This is a short test to measure voice cloning speed on Kaggle GPU."
else:
    text = script_path.read_text(encoding='utf-8')

chunks = vc.ultra_intelligent_chunking(text, 'ultra_fast')
print('Chunks to generate:', len(chunks))

audio_segments = []
chunk_times = []
start_all = time.time()
for i, chunk in enumerate(chunks):
    t0 = time.time()
    try:
        pitch, speed, energy = vc.process_markup_effects(chunk)
        clean = vc.clean_markup(chunk)
        audio = vc.ultra_fast_generate_chunk(clean, language='en', pitch_scale=pitch, speed_scale=speed, energy_scale=energy)
        audio_segments.append(audio)
    except Exception as e:
        print(f'Chunk {i} failed: {e}')
        continue
    t1 = time.time()
    chunk_times.append(t1 - t0)
    print(f'Chunk {i+1}/{len(chunks)} time: {t1-t0:.2f}s')

# Assemble
if not audio_segments:
    raise RuntimeError('No audio segments generated')

pause = np.zeros(int(0.1 * vc.tts.synthesizer.output_sample_rate), dtype=np.float32)
assembled = []
for i, seg in enumerate(audio_segments):
    assembled.append(seg)
    if i < len(audio_segments) - 1:
        assembled.append(pause)

final_audio = np.concatenate(assembled, axis=0)
# Normalize
final_audio = vc.ultra_fast_normalize_audio(final_audio)

out_path = Path('/kaggle/working/output/speech_kaggle.wav')
sf.write(str(out_path), final_audio, vc.tts.synthesizer.output_sample_rate)

total_time = time.time()-start_all
print('Total generation time:', total_time)
print('Audio duration (s):', len(final_audio)/vc.tts.synthesizer.output_sample_rate)
print('RTF:', total_time / (len(final_audio)/vc.tts.synthesizer.output_sample_rate))
print('Chunk times stats:', min(chunk_times), sum(chunk_times)/len(chunk_times), max(chunk_times))
print('Output written to:', out_path)

## 9) Save artifacts and create downloadable archive
Zip the `cache` and `output` directories so you can download the generated audio and cached models from the Kaggle UI.

In [ ]:
# Cell: Zip cache and output for download
import os

archive = '/kaggle/working/voice_clone_run.zip'
# Use zip to archive output and cache
if os.path.exists('/kaggle/working/cache') or os.path.exists('/kaggle/working/output'):
    try:
        !zip -r -q {archive} /kaggle/working/cache /kaggle/working/output
        print('Created archive:', archive)
    except Exception as e:
        print('Zip failed:', e)
else:
    print('Nothing to archive (cache/output missing)')

## 10) Quick reproducible test run
This final section contains a concise, reproducible test: upload a voice sample to `/kaggle/working/voice_sample.wav` and run the precompute + generation cells above. It will produce `/kaggle/working/output/speech_kaggle.wav` and a zip archive.